# Week 1 Lab, Part 2: The trajectory detective

In Part 1 the ligand's center moved by about 33 Å from where it started. But *moved relative to what?* In this part you investigate the same trajectory the way a detective would: **when** does the ligand leave its binding pocket, **what does it touch** on the way out, **what does the protein do** meanwhile, and **does the clock in the file mean what it seems to mean?**

**The rules of the case**

1. **Every answer needs a check:** a second, independent way to get the same number, or a reference value that cannot be faked. A number without a check is a claim, not a result.
2. **Predict first.** Write your guess in the *Predict first* cells before you run the code. A wrong guess is fine; an unreconciled one is not.
3. **A number needs a unit and a definition.** "The ligand leaves at frame 664" is incomplete until it says what "leaves" means.
4. **Figures** have no title, and axis labels are in sentence case with the unit in parentheses.
5. **AI assistants** are allowed under the course policy. Log what you asked, what was wrong and what you changed (see the last section).

**How to work.** Exactly as in Part 1: run cells top to bottom with **Shift+Enter**; cells marked **TODO** need your code; a cell whose first comment is `# check` is a test that must pass before you move on, and its message tells you what is wrong. Every section starts with **what you will compute** and a table of the arrays involved, because the shapes tell you what to do. When a check fails, print the shapes of your arrays first.

**Units.** mdtraj stores coordinates in **nanometers (nm)**; 1 nm = 10 Å. A name ending in `_nm` holds nanometers and `_A` holds angstroms.

### Setup: load the trajectory

Run the next cell. The `python` line should point into the course environment, as in Part 1. The cell also makes a **backup** of the coordinates, `xyz_original`: some mdtraj functions quietly change the data they are given, and later in this lab you will test that yours is still intact.

In [ ]:
import os, sys, platform
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import mdtraj as md

# Folder that holds sEH_nowater.pdb and sEH_unbinding.dcd: the ML4MD_DATA variable if it is set, otherwise this folder.
DATA_DIR = Path(os.environ.get("ML4MD_DATA", "."))
print("python :", platform.python_version(), "from", sys.executable)
print("numpy  :", np.__version__, "| mdtraj:", md.__version__)

traj = md.load(DATA_DIR / "sEH_unbinding.dcd", top=DATA_DIR / "sEH_nowater.pdb")
xyz = traj.xyz                      # (frames, atoms, 3), in nm
xyz_original = xyz.copy()           # a backup, so that we can test later that nothing changed the data
n_frames = traj.n_frames
print(traj)

### The cast: which atoms are which

The selections below are the ones you made in Part 1, given to you ready-made so that nothing in this part depends on your Part 1 answers. Seven sodium ions (`SOD`) are also in the file; they are neither ligand nor protein and we leave them out. The ligand has 45 atoms, of which 25 are not hydrogen (*heavy* atoms). One C-alpha atom per residue gives a coarse picture of the protein backbone.

In [ ]:
top = traj.topology
lig = top.select("resname '2RV'")                                         # the ligand: 45 atoms (note the quotes around 2RV)
protein = top.select("protein")                                           # every protein atom
is_heavy = np.array([atom.element.symbol != "H" for atom in top.atoms])   # True for every atom that is not hydrogen
lig_heavy = lig[is_heavy[lig]]                                            # the ligand atoms that are not hydrogen
protein_heavy = protein[is_heavy[protein]]                                # the protein atoms that are not hydrogen
ca = top.select("protein and name CA")                                    # one C-alpha atom per residue
protein_residues = [r for r in top.residues if r.is_protein]              # the protein's residues, in file order
lig_residue = top.atom(lig[0]).residue.index                              # the ligand's residue number in mdtraj's counting

print(f"{'group':18s} {'atoms':>6s}")
for name, group in [("ligand", lig), ("ligand, heavy", lig_heavy), ("protein", protein), ("protein, heavy", protein_heavy), ("protein C-alpha", ca)]:
    print(f"{name:18s} {len(group):6d}")
print("protein residues  :", len(protein_residues))

## 1. How far is the ligand from the protein?

**What you will compute.** One number for every frame: the distance between the **center of the ligand** and the **center of the protein**. This is a distance *inside* the system, so it does not care whether the whole system rotates or drifts in the box. (Part 1's displacement from frame 0 does care; Section 2 shows why that matters.)

**Worked example.** If the ligand center is at (1, 2, 2) nm and the protein center is at (0, 0, 0) nm, the arrow between them is (1, 2, 2) nm and its length is $\sqrt{1^2 + 2^2 + 2^2} = 3$ nm, which is 30 Å.

**Two ways, as the rules require.** You will get the centers once from mdtraj and once from numpy, and the answers must agree.

| name | what it holds | shape |
|:--|:--|:--|
| `lig_center_md`, `lig_center_np` | the ligand center in every frame (nm) | `(1107, 3)` |
| `protein_center_md`, `protein_center_np` | the protein center in every frame (nm) | `(1107, 3)` |
| `sep_A` | the distance between the two centers in every frame, in angstroms | `(1107,)` |

**Predict first.** In frame 0 the ligand sits in its pocket inside the protein. Will the distance between the two centers be about 1 Å, 9 Å or 30 Å? And in the last frame? (A protein like this one is roughly 4 to 5 nm across.)

My prediction: 

**Way 1: mdtraj.** `md.compute_center_of_geometry` takes a trajectory and returns the center of every frame. To measure only the ligand, first cut the trajectory down to the ligand's atoms with `traj.atom_slice(lig)`. The cell below does the cutting for you and prints the documentation of the function.

In [ ]:
lig_traj = traj.atom_slice(lig)                 # a new, smaller trajectory with only the ligand's 45 atoms
protein_traj = traj.atom_slice(protein)         # ... and one with only the protein's atoms
print("lig_traj.xyz.shape    :", lig_traj.xyz.shape, "= (frames, ligand atoms, x/y/z)")
print("protein_traj.xyz.shape:", protein_traj.xyz.shape)
help(md.compute_center_of_geometry)

In [ ]:
lig_center_md = None        # TODO: md.compute_center_of_geometry applied to lig_traj
protein_center_md = None    # TODO: the same function applied to protein_traj

# check: a plain Python loop over the atoms of one frame is a third, independent way
def center_by_loop(frame, atoms):
    total = np.zeros(3)
    for i in atoms:
        total += xyz_original[frame, i, :]
    return total / len(atoms)

assert lig_center_md is not None and protein_center_md is not None, "replace the Nones with your calls"
print("lig_center_md.shape:", lig_center_md.shape, "| wanted:", (n_frames, 3))
assert lig_center_md.shape == (n_frames, 3) and protein_center_md.shape == (n_frames, 3), "one (x, y, z) per frame"
for f in (0, 664, n_frames - 1):
    assert np.allclose(lig_center_md[f], center_by_loop(f, lig), atol=1e-4), f"the ligand center in frame {f} disagrees with the loop"
    assert np.allclose(protein_center_md[f], center_by_loop(f, protein), atol=1e-4), f"the protein center in frame {f} disagrees with the loop"
print("the mdtraj centers agree with the plain loop in frames 0, 664 and", n_frames - 1)

**Way 2: numpy.** In Part 1 you averaged `xyz[:, lig, :]` over its atoms axis to get the ligand center in every frame. Do the same for the ligand and for the protein (`xyz[:, protein, :]`, shape `(1107, 5052, 3)`). Remember: the axis you average over is the one that disappears from the shape.

In [ ]:
lig_center_np = None        # TODO: the average of xyz[:, lig, :] over the atoms axis
protein_center_np = None    # TODO: the same for xyz[:, protein, :]

# check
assert lig_center_md is not None, "do the mdtraj cell above first"
assert lig_center_np is not None and protein_center_np is not None, "replace the Nones with your calculations"
print("lig_center_np.shape:", lig_center_np.shape, "| wanted:", (n_frames, 3))
assert lig_center_np.shape == (n_frames, 3) and protein_center_np.shape == (n_frames, 3), "which axis has to disappear?"
assert np.allclose(lig_center_np, lig_center_md, atol=1e-4), "ligand: numpy and mdtraj disagree"
assert np.allclose(protein_center_np, protein_center_md, atol=1e-4), "protein: numpy and mdtraj disagree"
print("numpy and mdtraj agree in every frame, for the ligand and for the protein")

**The distance between the two centers.** Step 1, by hand for frame 5 (the cell below uses the plain loop from the check, so it does not depend on your arrays): the arrow from the protein center to the ligand center, and its length.

In [ ]:
arrow5 = center_by_loop(5, lig) - center_by_loop(5, protein)       # (3,): the arrow from the protein center to the ligand center, in nm
print("arrow5:", arrow5, "nm   shape", arrow5.shape)
print("length of arrow5:", np.linalg.norm(arrow5) * 10, "A")

Step 2, all frames at once: subtract the protein centers from the ligand centers (both have shape `(1107, 3)`, so this gives one arrow per frame), then take the length of every arrow. The unit conversion is the same as in Part 1: nm times 10 gives Å.

In [ ]:
sep_A = None    # TODO: the length of (lig_center_np - protein_center_np) in every frame, in angstroms

# check
assert sep_A is not None, "replace None above with your calculation"
print("sep_A.shape:", sep_A.shape, "| wanted:", (n_frames,))
assert sep_A.shape == (n_frames,), "one distance per frame: which axis of the (1107, 3) arrows has to disappear?"
assert abs(sep_A[5] - np.linalg.norm(arrow5) * 10) < 1e-2, "frame 5 disagrees with the by-hand value above"
sep_md_A = 10 * np.linalg.norm(lig_center_md - protein_center_md, axis=1)      # the same distance from the mdtraj centers
assert np.allclose(sep_A, sep_md_A, atol=1e-2), "the numpy and mdtraj centers give different distances"
print(f"center distance: frame 0 {sep_A[0]:.2f} A | frame 664 {sep_A[664]:.2f} A | last frame {sep_A[-1]:.2f} A")

**Two definitions of "the center".** The geometric center treats every atom alike. The **center of mass** counts heavy atoms more. mdtraj has both. The cell below compares them.

In [ ]:
lig_com = md.compute_center_of_mass(lig_traj)               # center of mass: heavier atoms count more
protein_com = md.compute_center_of_mass(protein_traj)
sep_com_A = 10 * np.linalg.norm(lig_com - protein_com, axis=1)
print(f"frame 0: geometric centers {sep_A[0]:.2f} A | centers of mass {sep_com_A[0]:.2f} A")
print(f"largest difference between the two definitions over all frames: {np.abs(sep_A - sep_com_A).max():.2f} A")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(np.arange(n_frames), sep_A)
ax.set_xlabel("Frame number")
ax.set_ylabel("Ligand-protein center distance (Å)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); plt.show()

**Judgment.** The two definitions of "the center" give numbers that differ by up to about 2 Å. Which would you use, and what would you write next to the number so that a reader could reproduce it?

My answer: 

## 2. Does the protein move?

In Part 1 the ligand center's displacement from frame 0 reached about 33 Å. If the **whole protein** also moved or turned in the box, part of that displacement is not the ligand leaving its pocket. To find out, measure how much the protein moves **on its own**.

**What you will compute.** For every frame, the **RMSD** (root-mean-square deviation) of the protein's C-alpha atoms from where they were in frame 0: a typical distance, in Å, between each C-alpha atom and its frame-0 position.

$$
\mathrm{RMSD}(t) = \sqrt{\frac{1}{N} \sum_{i=1}^{N} \left| \mathbf{r}_i(t) - \mathbf{r}_i(0) \right|^2}
$$

Here $N = 317$ is the number of C-alpha atoms, $\mathbf{r}_i(t)$ is the position of C-alpha atom $i$ in frame $t$ (nm), and $\left| \cdot \right|$ is the length of an arrow. **In words:** for each atom, take the distance it has moved and square it; average those squares over all atoms; take the square root.

**Worked example.** Three atoms move by 1, 2 and 2 nm. The squares are 1, 4 and 4; their average is 3; the square root is $\sqrt{3} \approx 1.73$ nm.

**Two versions.** *Unfitted:* compare positions exactly as stored. *Fitted:* first turn and shift each frame, as a rigid body, to match frame 0 as well as possible, and then compare. Fitting removes the motion of the protein as a whole and leaves only changes of its shape.

| name | what it holds | shape |
|:--|:--|:--|
| `ca_xyz` | the positions of the 317 C-alpha atoms in every frame (nm) | `(1107, 317, 3)` |
| `rmsd_unfit_A` | the unfitted RMSD in every frame (Å) | `(1107,)` |
| `rmsd_fit_A` | the fitted RMSD in every frame (Å) | `(1107,)` |

**Predict first.** At the last frame, will the unfitted RMSD be larger than, about the same as, or smaller than the fitted one? Roughly by what factor?

My prediction: 

**Step 1, one frame by hand** (frame 5). Run the cell and read each printed line: first the arrow each C-alpha atom has moved, then the squared length of each arrow, then the average and the root.

In [ ]:
ca_xyz = xyz_original[:, ca, :]          # (1107, 317, 3): only the C-alpha atoms
diff5 = ca_xyz[5] - ca_xyz[0]            # (317, 3): how far each C-alpha atom has moved by frame 5, as an arrow (nm)
sq5 = (diff5**2).sum(axis=1)             # (317,): the squared length of each arrow (nm squared)
rmsd5_A = np.sqrt(sq5.mean()) * 10       # average the squares, take the root, convert nm to angstroms
print("ca_xyz.shape:", ca_xyz.shape, "| diff5.shape:", diff5.shape, "| sq5.shape:", sq5.shape)
print(f"unfitted RMSD in frame 5: {rmsd5_A:.3f} A")

**Step 2, unfitted RMSD for all frames at once.** `ca_xyz - ca_xyz[0]` has shape `(1107, 317, 3)`. Square it, add the three directions (which axis?), and average over the atoms (which axis?).

In [ ]:
rmsd_unfit_A = None   # TODO: the same calculation for every frame at once, in angstroms

# check: frame 5 equals the by-hand value; the last frame equals a plain loop over atoms
assert rmsd_unfit_A is not None, "replace None above with your calculation"
print("rmsd_unfit_A.shape:", rmsd_unfit_A.shape, "| wanted:", (n_frames,))
assert rmsd_unfit_A.shape == (n_frames,), "one RMSD per frame"
assert rmsd_unfit_A[0] == 0.0, "frame 0 compared with itself must be 0"
assert abs(rmsd_unfit_A[5] - rmsd5_A) < 1e-3, "frame 5 disagrees with the by-hand value above"
total = 0.0
for k in range(len(ca)):
    total += np.sum((ca_xyz[-1, k, :] - ca_xyz[0, k, :])**2)
assert abs(rmsd_unfit_A[-1] - 10 * np.sqrt(total / len(ca))) < 1e-3, "the last frame disagrees with the plain loop"
print(f"unfitted RMSD: frame 5 {rmsd_unfit_A[5]:.2f} A | frame 664 {rmsd_unfit_A[664]:.2f} A | last frame {rmsd_unfit_A[-1]:.2f} A")

**Step 3, what fitting does.** A *fit* (or superposition) turns and shifts each frame as a rigid body so that it matches frame 0 as closely as possible. `traj.superpose(reference, frame, atom_indices)` does this **in place**, on the trajectory it is called on, so we call it on a **copy**, `traj[:]`, and never on `traj` itself. The cell below fits the copy and then applies the same RMSD formula to it. This is the second way, used for the check in the next step.

In [ ]:
fitted_copy = traj[:]                                         # a copy: fitting changes the object it is called on
fitted_copy.superpose(traj, 0, atom_indices=ca)              # turn and shift every frame of the copy to match frame 0 (C-alpha atoms)
fit_ca = fitted_copy.xyz[:, ca, :]
rmsd_fit_check_A = np.sqrt(((fit_ca - fit_ca[0])**2).sum(axis=2).mean(axis=1)) * 10    # the same RMSD formula, now after fitting
print(f"after fitting: frame 5 {rmsd_fit_check_A[5]:.2f} A | frame 664 {rmsd_fit_check_A[664]:.2f} A | last frame {rmsd_fit_check_A[-1]:.2f} A")
print("the original traj is untouched:", np.array_equal(traj.xyz, xyz_original))

**Step 4, mdtraj in one call.** `md.rmsd(target, reference, frame, atom_indices=...)` fits every frame of `target` to the reference frame and returns the RMSD in **nm**. **Two cautions.** Without `atom_indices` it uses *every* atom, including the ligand and the ions, which is not a protein RMSD. And in that case it also **moves your coordinates in place**. Always pass `atom_indices=ca` here; the check verifies that your data is still intact.

In [ ]:
rmsd_fit_A = None   # TODO: md.rmsd(traj, traj, 0, atom_indices=ca) returns nm; convert to angstroms

# check
assert rmsd_fit_A is not None, "replace None above with your calculation"
print("rmsd_fit_A.shape:", rmsd_fit_A.shape, "| wanted:", (n_frames,))
assert rmsd_fit_A.shape == (n_frames,), "one RMSD per frame"
assert np.array_equal(traj.xyz, xyz_original), "your call changed the coordinates in traj: pass atom_indices=ca so that only C-alpha atoms are used, then re-run the setup cell above to reload the data"
assert np.allclose(rmsd_fit_A, rmsd_fit_check_A, atol=1e-2), "md.rmsd and the fitted copy give different RMSDs: did you convert nm to angstroms?"
print(f"fitted RMSD: frame 5 {rmsd_fit_A[5]:.2f} A | frame 664 {rmsd_fit_A[664]:.2f} A | last frame {rmsd_fit_A[-1]:.2f} A | largest {rmsd_fit_A.max():.2f} A")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(np.arange(n_frames), rmsd_unfit_A, label="Without fitting")
ax.plot(np.arange(n_frames), rmsd_fit_A, label="After fitting")
ax.set_xlabel("Frame number")
ax.set_ylabel("C-alpha RMSD from frame 0 (Å)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); plt.show()

**Back to Part 1's plot.** The cell below recomputes the ligand center's displacement from frame 0 twice: as stored (the Part 1 plot), and after fitting the protein's C-alpha atoms onto frame 0, which removes the motion of the protein as a whole.

In [ ]:
ligand_center_by_frame = xyz_original[:, lig, :].mean(axis=1)                  # the ligand center, as in Part 1 (nm)
ligand_center_fitted = fitted_copy.xyz[:, lig, :].mean(axis=1)                 # the ligand center after fitting the protein onto frame 0
disp_unfit_A = 10 * np.linalg.norm(ligand_center_by_frame - ligand_center_by_frame[0], axis=1)
disp_fit_A = 10 * np.linalg.norm(ligand_center_fitted - ligand_center_fitted[0], axis=1)
print(f"largest displacement of the ligand center from frame 0: without fitting {disp_unfit_A.max():.2f} A | after fitting {disp_fit_A.max():.2f} A")
print(f"largest difference between the two at the same frame: {np.abs(disp_unfit_A - disp_fit_A).max():.2f} A")

**Judgment.** The protein's own shape change (fitted RMSD) stays near 1 Å, but its unfitted RMSD reaches 12 Å: the protein as a whole turns in the box. Yet the ligand's displacement changes by only a few Å when you fit. What does this tell you about the Part 1 plot? Which measure would you quote for "how far has the ligand gone": Part 1's displacement, the fitted displacement, or Section 1's center-to-center distance? Why?

My answer: 

## 3. When does the ligand leave?

**What you will compute.** For every frame, the **gap**: the smallest distance between any heavy atom of the ligand and any heavy atom of the protein, in Å. It is the distance between the ligand's closest atom and the protein's closest atom, like the gap between a hand and the wall it is touching. A hydrogen bond between two heavy atoms is about 2.8 Å long.

mdtraj can give, for every frame, the closest distance between the ligand and **each protein residue** (`md.compute_contacts` with `scheme="closest-heavy"`). The smallest of those 317 numbers in a frame is the gap. The cell below makes the table; it is in **nm**, because everything in mdtraj is.

| name | what it holds | shape |
|:--|:--|:--|
| `res_dist_nm` | for every frame and every protein residue, the closest distance between a ligand heavy atom and a heavy atom of that residue (nm) | `(1107, 317)` |
| `gap_A` | the smallest value in each row of `res_dist_nm`, in angstroms | `(1107,)` |

In [ ]:
pairs = [[lig_residue, r.index] for r in protein_residues]          # the ligand's residue paired with every protein residue
res_dist_nm, pair_list = md.compute_contacts(traj, pairs, scheme="closest-heavy", periodic=False)   # periodic=False: the file has no box
print("res_dist_nm.shape:", res_dist_nm.shape, "= (frames, protein residues), in nm")
print("column 0 is", protein_residues[0], "| the last column is", protein_residues[-1])
print("frame 0, the five smallest entries (nm):", np.sort(res_dist_nm[0])[:5])

**Predict first.** In frame 0 the ligand is bound. Will its gap to the protein be about 0.3 Å, 3 Å or 30 Å? And will the gap in the last frame be smaller or larger than in frame 0?

My prediction: 

**The second way, given.** For the check you need a way to get the gap that does not use `md.compute_contacts`. This one compares every ligand heavy atom with every protein heavy atom: `a[:, None, :] - b[None, :, :]` is the broadcasting trick from Part 1's pairwise-distance table, with shape `(25, 2560, 3)`. It is slow, so we use it only for checks. The cell below applies it to every frame, which takes a few seconds.

In [ ]:
def gap_by_brute_force(frame):
    a = xyz_original[frame, lig_heavy, :]                              # (25, 3): the ligand's heavy atoms in this frame
    b = xyz_original[frame, protein_heavy, :]                          # (2560, 3): the protein's heavy atoms
    pair = np.linalg.norm(a[:, None, :] - b[None, :, :], axis=2)       # (25, 2560): every ligand-protein pair
    return pair.min() * 10                                             # the smallest pair distance, in angstroms

gap_ref_A = np.array([gap_by_brute_force(t) for t in range(n_frames)])
print("gap_ref_A.shape:", gap_ref_A.shape, "| frame 0:", round(float(gap_ref_A[0]), 2), "A")

In [ ]:
gap_A = None    # TODO: the smallest value in each row of res_dist_nm (which axis?), in angstroms

# check
assert gap_A is not None, "replace None above with your calculation"
print("gap_A.shape:", gap_A.shape, "| wanted:", (n_frames,))
assert gap_A.shape == (n_frames,), "one gap per frame: take the smallest over the residues"
assert np.allclose(gap_A, gap_ref_A, atol=1e-2), "gap_A disagrees with the brute-force gap: did you convert nm to angstroms?"
print(f"gap: frame 0 {gap_A[0]:.2f} A | frame 664 {gap_A[664]:.2f} A | last frame {gap_A[-1]:.2f} A")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(np.arange(n_frames), gap_A)
ax.axhline(4.5, linestyle="--", color="gray")
ax.text(20, 5.0, "4.5 Å cutoff", color="gray")
ax.set_xlabel("Frame number")
ax.set_ylabel("Closest ligand-protein distance (Å)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); plt.show()

**"The frame where the ligand leaves" needs a definition.** Pick a cutoff (4.5 Å is a common choice for "in contact": a little more than a hydrogen bond or a close van der Waals contact). Two definitions:

* **A:** the first frame in which the gap is larger than the cutoff.
* **B (lost for good):** one more than the last frame in which the gap was at or below the cutoff, so the ligand never touches again.

You will compute both for five cutoffs and see how much the answer depends on the choice. A new piece of Python first: `a if condition else b` gives `a` when the condition is true and `b` otherwise.

In [ ]:
frames_seen = np.array([])                                          # an empty array: no frames
print(frames_seen[0] if len(frames_seen) > 0 else None)             # the condition is false, so this prints None
frames_seen = np.array([7, 9])
print(frames_seen[0] if len(frames_seen) > 0 else None)             # the condition is true, so this prints the first entry, 7

In [ ]:
cutoffs_A = [3.5, 4.5, 6.0, 8.0, 10.0]
table = {}                                              # cutoff -> (first frame above, frame after the last contact)
for cutoff in cutoffs_A:
    above = np.where(gap_A > cutoff)[0]                 # all frames in which the gap is larger than the cutoff
    within = np.where(gap_A <= cutoff)[0]               # all frames in which the ligand is within the cutoff
    first_above = None     # TODO (definition A): the first number in `above`, or None if `above` is empty
    lost_for_good = None   # TODO (definition B): one more than the last number in `within`, or None if that last number is the last frame of the trajectory
    table[cutoff] = (first_above, lost_for_good)

print(f"{'cutoff (A)':>10s} {'A: first frame above':>22s} {'B: lost for good at':>21s}")
for cutoff, (a, b) in table.items():
    print(f"{cutoff:10.1f} {str(a):>22s} {str(b):>21s}")

# check: plain loops over the brute-force gaps
def first_above_ref(cutoff):
    for t in range(n_frames):
        if gap_ref_A[t] > cutoff:
            return t
    return None
def lost_for_good_ref(cutoff):
    last = None
    for t in range(n_frames):
        if gap_ref_A[t] <= cutoff:
            last = t
    return None if last == n_frames - 1 else last + 1
for cutoff, (a, b) in table.items():
    assert a == first_above_ref(cutoff), f"cutoff {cutoff}: definition A should be {first_above_ref(cutoff)}, your table has {a}"
    assert b == lost_for_good_ref(cutoff), f"cutoff {cutoff}: definition B should be {lost_for_good_ref(cutoff)}, your table has {b}"
print("the table agrees with the plain loops")

**Judgment (graded on your reasoning, not on a single right answer).** Complete the sentences.

1. The ligand leaves at frame ___ (definition ___, cutoff ___ Å).
2. With the other definitions in the table it would be anywhere from frame ___ to frame ___ (or never).
3. I chose mine because: ___
4. Look back at the plot in Section 1: by frame 250 the center-to-center distance has grown from about 9 Å to about 24 Å, yet the gap in the plot above is still small. What is the ligand doing, and which of the two measures answers "is the ligand still bound?"

*Note on numbering:* mdtraj and numpy count frames from 0; VMD counts from 1, so the same frame is one higher in VMD.

My answer: 

## 4. Which residues does the ligand touch on the way out?

**What you will compute.** For each protein residue, the **fraction of the frames before the ligand leaves** in which that residue is in contact with the ligand. "In contact" means a distance of 4.5 Å or less, from the same `res_dist_nm` table as before. A fraction of 0.5 means the residue touched the ligand in half of those frames.

**Worked example.** If a residue is in contact in 3 of 4 frames, its fraction is 3/4 = 0.75. numpy does this directly: the average of `True`/`False` values is the fraction of `True`.

| name | what it holds | shape |
|:--|:--|:--|
| `in_contact` | `True` where the ligand is within the cutoff of the residue, for the frames before it leaves | `(664, 317)` |
| `contact_fraction` | for each residue, the fraction of those frames in which it is in contact | `(317,)` |

**Units.** `res_dist_nm` is in nm, so the cutoff must be too: 4.5 Å is 0.45 nm. Comparing a distance in nm with a cutoff written in Å is one of the most common silent mistakes with this kind of data.

In [ ]:
leave_frame = 664                                  # the frame from Section 3 (definition A, 4.5 A); use your own number if you chose differently
cutoff_nm = 4.5 / 10                               # the cutoff in the same unit as res_dist_nm: nm
in_contact = res_dist_nm[:leave_frame] < cutoff_nm # True/False: frames 0 to leave_frame - 1, every residue
print("in_contact.shape:", in_contact.shape, "= (frames before leaving, protein residues)")
print("residues in contact in frame 0:", int(in_contact[0].sum()))
print("the average of True/False values is the fraction of True:", np.array([True, False, True, True]).mean())

**Predict first.** How many of the 317 residues touch the ligand in frame 0: about 5, 20 or 100? Over all the frames before it leaves, will more residues touch it at some time, or fewer?

My prediction: 

In [ ]:
contact_fraction = None    # TODO: the fraction of frames (axis 0) in which each residue is in contact

# check: brute force for single residues, atom by atom, from the original coordinates
def residue_contact_fraction(column):
    atoms = [a.index for a in protein_residues[column].atoms if is_heavy[a.index]]
    hits = 0
    for t in range(leave_frame):
        pair = np.linalg.norm(xyz_original[t, lig_heavy, :][:, None, :] - xyz_original[t, atoms, :][None, :, :], axis=2)
        hits += pair.min() < cutoff_nm
    return hits / leave_frame

assert contact_fraction is not None, "replace None above with your calculation"
print("contact_fraction.shape:", contact_fraction.shape, "| wanted:", (len(protein_residues),))
assert contact_fraction.shape == (len(protein_residues),), "one fraction per residue: average over the frames"
tyr_column = [i for i, r in enumerate(protein_residues) if r.resSeq == 154][0]       # the residue with PDB number 154
for column in (int(np.argmax(contact_fraction)), tyr_column, 0):
    assert abs(contact_fraction[column] - residue_contact_fraction(column)) < 1e-9, f"residue {protein_residues[column]}: disagrees with the brute-force fraction"
print("the fractions agree with the brute-force calculation for three residues")

**Three numberings.** A residue has three different numbers, and mixing them up is a classic error:

* the **column** in your arrays: `protein_residues[k]` is column `k` (counting from 0);
* the **mdtraj index** `res.index`: the ligand is residue 0, so the first protein residue has index 1;
* the **PDB number** `res.resSeq`, which is what the structure file and VMD show.

Papers use a fourth: the **UniProt number**, counting along the full-length human sEH sequence (UniProt P34913). This simulation holds residues **231 to 547** of that sequence. Run the cell to see the first residue, then work out the number to add to a PDB number to get a UniProt number.

In [ ]:
first = protein_residues[0]
print(f"first protein residue: {first.name}, column 0, mdtraj index {first.index}, PDB number {first.resSeq}")
print("it is residue 231 of the full-length sequence (UniProt P34913)")

In [ ]:
offset = None    # TODO: the number to add to a PDB number (res.resSeq) to get the UniProt number

# check: against residues that every sEH paper names. Catalytic Asp335 and Asp496, His524; Tyr383 and Tyr466 hold the ligand.
assert offset is not None, "replace None above with your offset"
known = {335: "ASP", 383: "TYR", 466: "TYR", 496: "ASP", 524: "HIS"}
for number, name in known.items():
    found = [r.name for r in protein_residues if r.resSeq + offset == number]
    assert found == [name], f"with offset {offset}, the residue numbered {number} is {found}, but sEH has {name} there"
print(f"offset {offset} puts ASP at 335 and 496, TYR at 383 and 466, HIS at 524, as in the sEH literature")

In [ ]:
order = np.argsort(contact_fraction)[::-1]               # columns sorted from the most to the least contact
print(f"{'name':5s} {'column':>6s} {'mdtraj index':>12s} {'PDB number':>10s} {'UniProt number':>14s} {'fraction':>9s}")
for k in order[:12]:
    res = protein_residues[k]
    print(f"{res.name:5s} {k:6d} {res.index:12d} {res.resSeq:10d} {res.resSeq + offset:14d} {contact_fraction[k]:9.2f}")
print("residues that touch the ligand in at least one frame before it leaves:", int((contact_fraction > 0).sum()))

**Judgment.** (1) Which of the five catalytic residues in the check does the ligand touch before it leaves, and in what fraction of the frames? (Look them up in `contact_fraction` by their column.) (2) If you had printed `res.index + 229` instead of `res.resSeq + 229`, which residue would you have called "Asp335", and why would the check above have caught it?

My answer: 

## 5. Is the protein changing size?

**What you will compute.** For every frame, the protein's **radius of gyration** $R_g$: the typical distance of an atom from the center of the protein, in Å. A protein that unfolds or swells gets a larger $R_g$.

$$
R_g = \sqrt{\frac{1}{N} \sum_{i=1}^{N} \left| \mathbf{r}_i - \mathbf{r}_c \right|^2}
$$

Here $N = 5052$ is the number of protein atoms, $\mathbf{r}_i$ is the position of atom $i$ (nm) and $\mathbf{r}_c$ is the center of the protein (nm) in the same frame. **In words:** for each atom, square its distance from the center, average over all atoms, and take the square root. This is the same recipe as the RMSD, with the protein's own center in place of frame 0.

**Worked example.** Two atoms at $-1$ nm and $+1$ nm on the x axis have their center at 0. Each is 1 nm from it, so $R_g = \sqrt{(1 + 1)/2} = 1$ nm.

| name | what it holds | shape |
|:--|:--|:--|
| `protein_xyz` | the positions of the protein atoms in every frame (nm) | `(1107, 5052, 3)` |
| `rg_md_A`, `rg_np_A` | the radius of gyration in every frame, from mdtraj and from numpy (Å) | `(1107,)` |

In [ ]:
protein_xyz = xyz_original[:, protein, :]                         # (1107, 5052, 3): only the protein atoms
center5 = protein_xyz[5].mean(axis=0)                             # (3,): the protein center in frame 5
dist5 = np.linalg.norm(protein_xyz[5] - center5, axis=1)          # (5052,): each atom's distance from that center (nm)
rg5_A = np.sqrt((dist5**2).mean()) * 10                           # square, average, root; nm -> A
print("protein_xyz.shape:", protein_xyz.shape, "| dist5.shape:", dist5.shape)
print(f"radius of gyration in frame 5: {rg5_A:.3f} A")

**Predict first.** Will $R_g$ change by about 0.1 Å, 1 Å or 10 Å while the ligand leaves? Why?

My prediction: 

**Way 1: mdtraj.** `md.compute_rg(protein_traj)` returns $R_g$ for every frame, in nm. *Check its documentation:* it treats all atoms as equally heavy unless you give it `masses=`.

In [ ]:
rg_md_A = None    # TODO: md.compute_rg applied to protein_traj, converted to angstroms

# check
assert rg_md_A is not None, "replace None above with your calculation"
print("rg_md_A.shape:", rg_md_A.shape, "| wanted:", (n_frames,))
assert rg_md_A.shape == (n_frames,), "one value per frame"
assert abs(rg_md_A[5] - rg5_A) < 1e-2, "frame 5 disagrees with the by-hand value above: nm to angstroms?"
print(f"R_g from mdtraj: frame 5 {rg_md_A[5]:.3f} A | smallest {rg_md_A.min():.3f} A | largest {rg_md_A.max():.3f} A")

**Way 2: numpy, all frames at once.** Frame 5 by hand used `protein_xyz[5] - center5`, an array `(5052, 3)` minus a vector `(3,)`. For all frames, the centers have shape `(1107, 3)` but the atoms have shape `(1107, 5052, 3)`: the centers need an extra axis of length 1 in the middle, `center[:, None, :]` with shape `(1107, 1, 3)`, so that numpy stretches it over the 5052 atoms of each frame.

In [ ]:
rg_np_A = None    # TODO: the radius of gyration of every frame from protein_xyz, in angstroms
# hint: center = protein_xyz.mean(axis=1) has shape (1107, 3); subtract center[:, None, :]; square and sum x/y/z (axis=2);
#       average over atoms (axis=1); square root; nm -> A. (If numpy says the shapes 'could not be broadcast together', the center is missing its extra axis.)

# check
assert rg_md_A is not None, "do the mdtraj cell above first"
assert rg_np_A is not None, "replace None above with your calculation"
print("rg_np_A.shape:", rg_np_A.shape, "| wanted:", (n_frames,))
assert rg_np_A.shape == (n_frames,), "one value per frame"
assert abs(rg_np_A[5] - rg5_A) < 1e-2, "frame 5 disagrees with the by-hand value above"
assert np.allclose(rg_np_A, rg_md_A, atol=1e-2), "numpy and mdtraj disagree"
print("numpy and mdtraj agree in every frame")

In [ ]:
masses = np.array([atom.element.mass for atom in protein_traj.topology.atoms])    # the mass of each protein atom, in atomic mass units
rg_mass_A = 10 * md.compute_rg(protein_traj, masses=masses)                       # the mass-weighted radius of gyration
print(f"largest difference between the equal-weight and the mass-weighted R_g: {np.abs(rg_md_A - rg_mass_A).max():.3f} A")
print(f"R_g over the trajectory: smallest {rg_md_A.min():.2f} A, largest {rg_md_A.max():.2f} A, range {rg_md_A.max() - rg_md_A.min():.2f} A")

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(np.arange(n_frames), rg_md_A)
ax.set_xlabel("Frame number")
ax.set_ylabel("Protein radius of gyration (Å)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); plt.show()

**Judgment.** Does the protein change size while the ligand leaves? Put the range of $R_g$ next to the other numbers you have: the fitted RMSD and the ligand's distance from the protein. And: the equal-weight and mass-weighted values differ slightly; which would you report, and what would you write next to it?

My answer: 

## 6. The missing clock

So far everything has been in **frames**. A paper would say how long the unbinding took, in nanoseconds. mdtraj stores a time for every frame in `traj.time`, and the time between frames in `traj.timestep`. Run the cell and read what they say.

In [ ]:
print("traj.time[:5]   :", traj.time[:5])
print("traj.time[-3:]  :", traj.time[-3:])
print("traj.timestep   :", traj.timestep)
print("number of frames:", n_frames)

**Predict first.** Does `traj.time[664]` tell you that the ligand left after 664 ps, 664 fs, or something else? What would you test to find out?

My prediction: 

In [ ]:
time_is_frame_number = None    # TODO: True or False: is traj.time[i] equal to i for every frame i?

# check: a plain loop over the frames
expected = all(traj.time[i] == i for i in range(n_frames))
assert time_is_frame_number is not None, "replace None above with True or False, computed from traj.time"
assert time_is_frame_number == expected, "test it: compare traj.time with the numbers 0, 1, 2, ... one per frame"
print("time_is_frame_number =", time_is_frame_number)

**Graded question: how long did the ligand take to leave?** Answer each part in a sentence or two.

1. What does `traj.time` contain, and what is your evidence? (Use the check above.)
2. Which two numbers would you need to turn a frame number into nanoseconds, and where would you look for them?
3. Name one reason why, even with those two numbers, the length of this one trajectory would not tell you how long a ligand like this one usually stays bound.
4. State your answer to "when does the ligand leave?" using only what you can defend: a frame number, the definition you used, and what you do or do not know about the time.

My answer:

1. 
2. 
3. 
4. 

## 7. Audit an AI assistant

An assistant was asked: *"Using mdtraj, find when the ligand unbinds in sEH_unbinding.dcd, how long it takes, which residues it contacts before leaving, and how much the protein moves."* Its code is below, and it **runs without an error**. One of its five printed claims is right. The other four are wrong, each for a different reason.

Your job: find them by comparing with **your own results from Sections 1 to 6**, not by asking another assistant. The assistant works on its own copy of the data, so running this cell does not change yours.

In [ ]:
# ---- the assistant's answer (the thing under audit; do not edit) --------------------------------------
import numpy as np
import mdtraj as md

t = md.load(DATA_DIR / "sEH_unbinding.dcd", top=DATA_DIR / "sEH_nowater.pdb")
lig_atoms = t.topology.select("resname '2RV'")
lig_res = t.topology.atom(lig_atoms[0]).residue.index
prot_res = [r for r in t.topology.residues if r.is_protein]
cutoff = 4.5                                                   # contact cutoff in angstrom

# 1. When does the ligand unbind? Closest ligand-protein distance in every frame.
pairs = [[lig_res, r.index] for r in prot_res]
d, _ = md.compute_contacts(t, pairs, scheme="closest-heavy", periodic=False)    # (frames, residues)
closest = d.min(axis=1) * 10                                   # convert to angstrom
unbind = int(np.argmax(closest > cutoff))                      # first frame beyond the cutoff
print(f"Unbinding frame: {unbind}")

# 2. How long does it take? The trajectory time is stored in picoseconds.
print(f"Unbinding time: {t.time[unbind] / 1000:.2f} ns")

# 3. Which residues does the ligand contact before it leaves?
touched = (d[:unbind] < cutoff).any(axis=0)                    # within the cutoff in at least one frame
print(f"Residues contacted before unbinding: {touched.sum()}")
frac = (10 * d[:unbind] < cutoff).mean(axis=0)                 # fraction of frames in contact
top = np.argsort(frac)[::-1][:3]
print("Most contacted residues (UniProt numbering):", [(prot_res[k].name, prot_res[k].index + 229) for k in top])

# 4. How much does the protein move? RMSD from the first frame.
rmsd = md.rmsd(t, t, 0) * 10
print(f"Protein RMSD: up to {rmsd.max():.1f} A")

**Your audit.** Fill in one row per claim. For each wrong claim, say *which of your own results contradicts it*, what the mistake is, and how to fix it. A claim you think is right needs your evidence too.

| claim | right or wrong | the evidence from my own results | the mistake and the fix |
|:--|:--|:--|:--|
| 1. Unbinding frame |  |  |  |
| 2. Unbinding time |  |  |  |
| 3a. Number of residues contacted |  |  |  |
| 3b. Most contacted residues (UniProt numbers) |  |  |  |
| 4. Protein RMSD |  |  |  |

**Fix the claims.** For the three claims you found that have a numerical answer, give the corrected values, taken from your results. The check compares them with reference values that do not depend on your code.

In [ ]:
n_contacted_fixed = None           # TODO: how many residues are within 4.5 A of the ligand in at least one frame before it leaves? (Section 4)
protein_rmsd_max_fixed_A = None    # TODO: the largest C-alpha RMSD after fitting, in angstroms (Section 2)
top3_fixed = None                  # TODO: the three most-contacted residues with their UniProt numbers, as a list like [("PHE", 497), ...] (Section 4)

# check: against reference values computed from the given arrays
assert None not in (n_contacted_fixed, protein_rmsd_max_fixed_A, top3_fixed), "replace the Nones"
assert n_contacted_fixed == int(in_contact.any(axis=0).sum()), "the number of residues contacted at least once before leaving does not match"
assert abs(protein_rmsd_max_fixed_A - rmsd_fit_check_A.max()) < 0.05, "the largest fitted C-alpha RMSD does not match"
ref_fraction = in_contact.sum(axis=0) / in_contact.shape[0]
ref_top3 = {(protein_residues[k].name, protein_residues[k].resSeq + 229) for k in np.argsort(ref_fraction)[::-1][:3]}
assert {tuple(x) for x in top3_fixed} == ref_top3, f"the top three should be {sorted(ref_top3, key=lambda x: x[1])} (name, UniProt number)"
print("the three corrected values agree with the reference values")

## 8. Log it, and get ready to explain it

**AI-use log, entry 1** (you started it in Part 1; finish it now). Use the template `W1_AI_use_log_template.md` in this folder: what you asked an assistant (or "nothing"), what was wrong or misleading, how you found out, and what you changed. The audit in Section 7 is a good place to start: did the assistant you use make any of these four mistakes?

**Oral check-off (3 to 5 minutes, in lab).** An instructor will ask you to pick **one figure** from this notebook and **one line of code** and explain them. Be ready to say what the figure shows and in what units, how you know the number is right (which check), and what would change if a definition changed (the cutoff, fitting or not, the clock). Expect questions such as "where does this number come from?" and "what would you see if you had not converted from nm?".

**What to hand in for Week 1**

* This notebook with your predictions, your written answers and every check passing.
* The one-line **PROOF** output and the job ID from your ICER job (Part 0).
* Part 1's notebook.
* Your AI-use log, entry 1.